# Evaluation các model GRPO trên Kaggle

Notebook này tải lại từng model từ `Kaggle_data`, phân biệt theo đặc điểm trong `grpo.yaml`, rồi đánh giá trên hai dataset Hugging Face:
- `ChilleD/MultiArith` (test split)
- `apple/GSM-Symbolic` (test split)

Mỗi câu hỏi được đánh giá riêng:
- `MultiArith`: kiểm tra xem đáp án đúng xuất hiện trong output của model với điều kiện không nằm trong số dài hơn.
- `GSM-Symbolic`: tính cosine similarity TF-IDF giữa output của model và đáp án đúng.

Kết quả cuối cùng được so sánh theo điểm riêng cho từng dataset.

In [6]:
import os
import sys
import subprocess
from pathlib import Path

# Cài gói nếu cần thiết
def install_if_missing(packages):
    import importlib
    missing = []
    for pkg in packages:
        if importlib.util.find_spec(pkg) is None:
            missing.append(pkg)
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

install_if_missing(["peft", "datasets", "scikit_learn", "pyyaml"])

# Nếu dùng Qwen 3.5, cài lại transformers từ source mà không cập nhật dependencies khác.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-deps",
    "--force-reinstall",
    "git+https://github.com/huggingface/transformers.git@main",
    "-q",
])

import json
import re
import pandas as pd
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import yaml

# Dataset path trên Kaggle

dataset_path = Path("/kaggle/input/datasets/qucnguyntin/kaggle-data/Kaggle_data")
if not dataset_path.exists():
    dataset_path = Path("Kaggle_data")

print("dataset_path:", dataset_path)
print("exists:", dataset_path.exists())

model_dirs = sorted([p for p in dataset_path.iterdir() if p.is_dir()])
print("Found model folders:", [p.name for p in model_dirs])

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token_env = user_secrets.get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets.")
except ImportError:
    # Nếu không chạy trên Kaggle (ví dụ: Local, Colab), nó sẽ nhảy vào đây
    pass
except Exception as e:
    # Bắt lỗi trường hợp có thư viện nhưng bạn quên chưa đính kèm Secret vào notebook
    print(f"Could not load from Kaggle Secrets: {e}")

use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)



Installing missing packages: ['scikit_learn', 'pyyaml']
dataset_path: /kaggle/input/datasets/qucnguyntin/kaggle-data/Kaggle_data
exists: True
Found model folders: ['results-qwen_1', 'results_deepseek_1', 'results_deepseek_2', 'results_qwen_2']
Loaded HF token from Kaggle Secrets.
CUDA available: True


In [7]:
def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def model_signature(config: dict):
    parts = []
    base = config.get("model_name_or_path", "unknown").replace("/", "_")
    parts.append(base)
    dtype = config.get("torch_dtype") or config.get("dtype") or "unknown_dtype"
    parts.append(dtype)
    if config.get("bf16"):
        parts.append("bf16")
    if config.get("fp16"):
        parts.append("fp16")
    lr = config.get("learning_rate")
    if lr is not None:
        parts.append(f"lr{lr}")
    max_steps = config.get("max_steps")
    if max_steps is not None:
        parts.append(f"steps{max_steps}")
    max_completion = config.get("max_completion_length")
    if max_completion is not None:
        parts.append(f"maxc{max_completion}")
    return "__".join(parts)


def load_model_from_folder(model_dir: Path):
    grpo_path = model_dir / "grpo.yaml"
    config = load_yaml(grpo_path)
    signature = model_signature(config)
    if not signature:
        signature = model_dir.name

    tokenizer = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    except Exception as exc:
        print(f"Không load được tokenizer từ {model_dir}: {exc}")
        tokenizer = AutoTokenizer.from_pretrained(config.get("model_name_or_path"), trust_remote_code=True)

    base_model_id = config.get("model_name_or_path")
    device_map = "auto" if use_cuda else "cpu"
    torch_dtype = torch.float16 if use_cuda else torch.float32

    print(f"Loading base model {base_model_id} for {signature}")
    base_config = AutoConfig.from_pretrained(
        base_model_id,
        trust_remote_code=True,
        revision=config.get("model_revision", "main"),
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        config=base_config,
        trust_remote_code=True,
        device_map=device_map,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(
        base_model,
        model_dir,
        torch_dtype=torch_dtype,
        device_map=device_map,
    )
    model.eval()

    return signature, {
        "folder": model_dir.name,
        "path": str(model_dir),
        "config": config,
        "tokenizer": tokenizer,
        "model": model,
    }

loaded_models = {}
for model_dir in model_dirs:
    try:
        signature, info = load_model_from_folder(model_dir)
        if signature in loaded_models:
            signature = f"{signature}__{model_dir.name}"
        loaded_models[signature] = info
        globals()[signature] = info
    except Exception as exc:
        print(f"Lỗi khi load model từ {model_dir}: {exc}")

print("Loaded model keys:", list(loaded_models.keys()))

Loading base model Qwen/Qwen3.5-0.8B for Qwen_Qwen3.5-0.8B__float16__bf16__lr1e-06__steps20__maxc512


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading base model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B for deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__fp16__lr1e-05__steps100__maxc1024


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading base model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B for deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__bf16__lr1e-06__steps20__maxc1024


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading base model Qwen/Qwen3.5-0.8B for Qwen_Qwen3.5-0.8B__float16__bf16__lr2e-06__steps20__maxc1024


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded model keys: ['Qwen_Qwen3.5-0.8B__float16__bf16__lr1e-06__steps20__maxc512', 'deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__fp16__lr1e-05__steps100__maxc1024', 'deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__bf16__lr1e-06__steps20__maxc1024', 'Qwen_Qwen3.5-0.8B__float16__bf16__lr2e-06__steps20__maxc1024']


In [16]:
# Load Hugging Face datasets
datasets = {}
max_eval_samples = 15  # nếu muốn test nhanh, đặt giá trị ví dụ 50
for dataset_id in ["ChilleD/MultiArith", "apple/GSM-Symbolic"]:
    print(f"Loading dataset {dataset_id}")
    datasets[dataset_id] = load_dataset(dataset_id, split="test")
    print(dataset_id, "columns:", datasets[dataset_id].column_names)
    if max_eval_samples is not None:
        datasets[dataset_id] = datasets[dataset_id].select(range(min(max_eval_samples, len(datasets[dataset_id]))))

# Helpers

def normalize_text(text):
    if text is None:
        return ""
    return str(text).strip()


def contains_exact_numeric_answer(answer, text):
    answer = normalize_text(answer)
    if answer == "":
        return False
    escaped = re.escape(answer)
    pattern = rf"(?<!\d){escaped}(?!\d)"
    return bool(re.search(pattern, text))


def tfidf_similarity(answer, prediction):
    texts = [normalize_text(answer), normalize_text(prediction)]
    vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b")
    vecs = vectorizer.fit_transform(texts)
    if vecs.shape[1] == 0:
        return 0.0
    sim = cosine_similarity(vecs[0:1], vecs[1:2])[0, 0]
    return float(sim)


def generate_text(model, tokenizer, text, max_new_tokens=128):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def detect_question_answer(example):
    question = None
    answer = None
    for qkey in ["question", "Question", "input", "problem"]:
        if qkey in example:
            question = example[qkey]
            break
    for akey in ["answer", "Answer", "labels", "final_ans", "target"]:
        if akey in example:
            answer = example[akey]
            break
    return question, answer

print("Ready to evaluate.")

Loading dataset ChilleD/MultiArith
ChilleD/MultiArith columns: ['question', 'final_ans']
Loading dataset apple/GSM-Symbolic
apple/GSM-Symbolic columns: ['id', 'instance', 'question', 'answer', 'original_id', 'original_question', 'original_answer', 'canary']
Ready to evaluate.


In [17]:
# Transform datasets to match training format
system_prompt = """A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \\boxed{{}} . The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. Note that respond by English, NOT use other languages."""

def format_question(question):
    return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n<think>"

# Transform MultiArith dataset
transformed_multi = []
for example in datasets["ChilleD/MultiArith"]:
    question, answer = detect_question_answer(example)
    if question is not None:
        formatted_question = format_question(question)
        transformed_multi.append({
            "original_question": question,
            "formatted_question": formatted_question,
            "answer": answer,
        })

# Transform GSM-Symbolic dataset
transformed_gsm = []
for example in datasets["apple/GSM-Symbolic"]:
    question, answer = detect_question_answer(example)
    if question is not None:
        formatted_question = format_question(question)
        transformed_gsm.append({
            "original_question": question,
            "formatted_question": formatted_question,
            "answer": answer,
        })

print(f"Transformed {len(transformed_multi)} MultiArith examples")
print(f"Transformed {len(transformed_gsm)} GSM-Symbolic examples")
print("Sample formatted question:")
print(transformed_multi[0]["formatted_question"] if transformed_multi else "No examples")

Transformed 15 MultiArith examples
Transformed 15 GSM-Symbolic examples
Sample formatted question:
<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. Note that respond by English, NOT use other languages.<|im_end|>
<|im_start|>user
 Paige had 11 songs on her mp3 player. If she deleted 9 old songs from it and then added 8 new songs, how many songs does she have on her mp3 player? <|im_end|>
<|im_start|>assistant
<think>


In [18]:
evaluation_results = []

for model_key, meta in loaded_models.items():
    print("Evaluating model:", model_key)
    model = meta["model"]
    tokenizer = meta["tokenizer"]

    # MultiArith
    multi_dataset = transformed_multi
    multi_correct = 0
    multi_total = 0
    multi_records = []
    for example in multi_dataset:
        formatted_question = example["formatted_question"]
        answer = example["answer"]
        if formatted_question is None or answer is None:
            continue
        prediction = generate_text(model, tokenizer, formatted_question)
        correct = contains_exact_numeric_answer(answer, prediction)
        multi_records.append({
            "question": example["original_question"],
            "formatted_question": formatted_question,
            "answer": answer,
            "prediction": prediction,
            "correct": correct,
        })
        multi_correct += int(correct)
        multi_total += 1

    multi_score = float(multi_correct / multi_total) if multi_total else 0.0

    # GSM-Symbolic
    gsm_dataset = transformed_gsm
    gsm_score_sum = 0.0
    gsm_total = 0
    gsm_records = []
    for example in gsm_dataset:
        formatted_question = example["formatted_question"]
        answer = example["answer"]
        if formatted_question is None or answer is None:
            continue
        prediction = generate_text(model, tokenizer, formatted_question)
        sim = tfidf_similarity(answer, prediction)
        gsm_records.append({
            "question": example["original_question"],
            "formatted_question": formatted_question,
            "answer": answer,
            "prediction": prediction,
            "tfidf_score": sim,
        })
        gsm_score_sum += sim
        gsm_total += 1

    gsm_score = float(gsm_score_sum / gsm_total) if gsm_total else 0.0

    evaluation_results.append({
        "model_key": model_key,
        "folder": meta["folder"],
        "multi_arith_accuracy": multi_score,
        "multi_arith_correct": multi_correct,
        "multi_arith_total": multi_total,
        "gsm_symbolic_tfidf": gsm_score,
        "gsm_symbolic_total": gsm_total,
        "multi_examples": multi_records,
        "gsm_examples": gsm_records,
    })

results_df = pd.DataFrame([
    {
        "model_key": r["model_key"],
        "folder": r["folder"],
        "multi_arith_accuracy": r["multi_arith_accuracy"],
        "gsm_symbolic_tfidf": r["gsm_symbolic_tfidf"],
        "multi_arith_total": r["multi_arith_total"],
        "gsm_symbolic_total": r["gsm_symbolic_total"],
    }
    for r in evaluation_results
])

print("Evaluation complete.")
results_df

Evaluating model: Qwen_Qwen3.5-0.8B__float16__bf16__lr1e-06__steps20__maxc512
Evaluating model: deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__fp16__lr1e-05__steps100__maxc1024
Evaluating model: deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__bf16__lr1e-06__steps20__maxc1024
Evaluating model: Qwen_Qwen3.5-0.8B__float16__bf16__lr2e-06__steps20__maxc1024
Evaluation complete.


,model_key,folder,multi_arith_accuracy,gsm_symbolic_tfidf,multi_arith_total,gsm_symbolic_total
0,Qwen_Qwen3.5-0.8B__float16__bf16__lr1e-06__steps20__maxc512,results-qwen_1,0.266667,0.527093,15,15
1,deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__fp16__lr1e-05__steps100__maxc1024,results_deepseek_1,0.066667,0.022799,15,15
2,deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__bf16__lr1e-06__steps20__maxc1024,results_deepseek_2,0.066667,0.022559,15,15
3,Qwen_Qwen3.5-0.8B__float16__bf16__lr2e-06__steps20__maxc1024,results_qwen_2,0.266667,0.527093,15,15


In [20]:
# Hiển thị kết quả chi tiết và tóm tắt
pd.set_option("display.max_colwidth", 300)
print("=== Summary ===")
display(results_df.sort_values(["multi_arith_accuracy", "gsm_symbolic_tfidf"], ascending=False))

for record in evaluation_results:
    print(f"\n--- {record['model_key']} ---")
    print(f"MultiArith accuracy: {record['multi_arith_accuracy']:.4f} ({record['multi_arith_correct']}/{record['multi_arith_total']})")
    print(f"GSM-Symbolic avg TF-IDF score: {record['gsm_symbolic_tfidf']:.4f}")
    print("Example outputs:")
    display(pd.DataFrame(record["multi_examples"][:3]))
    display(pd.DataFrame(record["gsm_examples"][:3]))

=== Summary ===


,model_key,folder,multi_arith_accuracy,gsm_symbolic_tfidf,multi_arith_total,gsm_symbolic_total
0,Qwen_Qwen3.5-0.8B__float16__bf16__lr1e-06__steps20__maxc512,results-qwen_1,0.266667,0.527093,15,15
3,Qwen_Qwen3.5-0.8B__float16__bf16__lr2e-06__steps20__maxc1024,results_qwen_2,0.266667,0.527093,15,15
1,deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__fp16__lr1e-05__steps100__maxc1024,results_deepseek_1,0.066667,0.022799,15,15
2,deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B__float16__bf16__lr1e-06__steps20__maxc1024,results_deepseek_2,0.066667,0.022559,15,15



--- Qwen_Qwen3.5-0.8B__float16__bf16__lr1e-06__steps20__maxc512 ---
MultiArith accuracy: 0.2667 (4/15)
GSM-Symbolic avg TF-IDF score: 0.5271
Example outputs:


,question,formatted_question,answer,prediction,correct
0,"Paige had 11 songs on her mp3 player. If she deleted 9 old songs from it and then added 8 new songs, how many songs does she have on her mp3 player?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",10,"system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",True
1,"There were 39 girls and 4 boys trying out for the schools basketball team. If only 26 of them got called back, how many students didn't make the cut?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",17,"system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",False
2,"Faye had 34 coloring books. If she gave away 3 of them, but then bought 48 more, how many would she have total?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",79,"system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",True


,question,formatted_question,answer,prediction,tfidf_score
0,Rania saw a 210-foot whale with 7 72-inch remoras attached to it. What percentage of the whale's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 72 inches/remora * 7 remoras = 504 inches.\nThen divide that number by 12 to convert it to feet: 504 inches / 12 inches/foot = 42 foot.\nThen divide the combined remora length in feet by the whale's length and multiply by 100% to express ...","system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",0.529875
1,Olivia saw a 10-foot dolphin with 8 3-inch remoras attached to it. What percentage of the dolphin's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 3 inches/remora * 8 remoras = 24 inches.\nThen divide that number by 12 to convert it to feet: 24 inches / 12 inches/foot = 2 foot.\nThen divide the combined remora length in feet by the dolphin's length and multiply by 100% to express th...","system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",0.537106
2,Jamal saw a 380-foot whale with 8 57-inch remoras attached to it. What percentage of the whale's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 57 inches/remora * 8 remoras = 456 inches.\nThen divide that number by 12 to convert it to feet: 456 inches / 12 inches/foot = 38 foot.\nThen divide the combined remora length in feet by the whale's length and multiply by 100% to express ...","system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",0.517750


...

,question,formatted_question,answer,prediction,correct
0,"Paige had 11 songs on her mp3 player. If she deleted 9 old songs from it and then added 8 new songs, how many songs does she have on her mp3 player?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",10,"<|im_start|>systemAconversationbetweenUserandAssistant.Theuserasksaquestion,andtheAssistantsolvesit.Theassistantfirstthinksaboutthereasoningprocessinthemindandthenprovidestheuserwiththeanswer,andputyourfinalanswerwithin\boxed{{}}.Thereasoningprocessandanswerareenclosedwithin<think></think>and<an...",False
1,"There were 39 girls and 4 boys trying out for the schools basketball team. If only 26 of them got called back, how many students didn't make the cut?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",17,"<|im_start|>systemAconversationbetweenUserandAssistant.Theuserasksaquestion,andtheAssistantsolvesit.Theassistantfirstthinksaboutthereasoningprocessinthemindandthenprovidestheuserwiththeanswer,andputyourfinalanswerwithin\boxed{{}}.Thereasoningprocessandanswerareenclosedwithin<think></think>and<an...",True
2,"Faye had 34 coloring books. If she gave away 3 of them, but then bought 48 more, how many would she have total?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",79,"<|im_start|>systemAconversationbetweenUserandAssistant.Theuserasksaquestion,andtheAssistantsolvesit.Theassistantfirstthinksaboutthereasoningprocessinthemindandthenprovidestheuserwiththeanswer,andputyourfinalanswerwithin\boxed{{}}.Thereasoningprocessandanswerareenclosedwithin<think></think>and<an...",False


,question,formatted_question,answer,prediction,tfidf_score
0,Rania saw a 210-foot whale with 7 72-inch remoras attached to it. What percentage of the whale's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 72 inches/remora * 7 remoras = 504 inches.\nThen divide that number by 12 to convert it to feet: 504 inches / 12 inches/foot = 42 foot.\nThen divide the combined remora length in feet by the whale's length and multiply by 100% to express ...","<|im_start|>systemAconversationbetweenUserandAssistant.Theuserasksaquestion,andtheAssistantsolvesit.Theassistantfirstthinksaboutthereasoningprocessinthemindandthenprovidestheuserwiththeanswer,andputyourfinalanswerwithin\boxed{{}}.Thereasoningprocessandanswerareenclosedwithin<think></think>and<an...",0.024765
1,Olivia saw a 10-foot dolphin with 8 3-inch remoras attached to it. What percentage of the dolphin's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 3 inches/remora * 8 remoras = 24 inches.\nThen divide that number by 12 to convert it to feet: 24 inches / 12 inches/foot = 2 foot.\nThen divide the combined remora length in feet by the dolphin's length and multiply by 100% to express th...","<|im_start|>systemAconversationbetweenUserandAssistant.Theuserasksaquestion,andtheAssistantsolvesit.Theassistantfirstthinksaboutthereasoningprocessinthemindandthenprovidestheuserwiththeanswer,andputyourfinalanswerwithin\boxed{{}}.Thereasoningprocessandanswerareenclosedwithin<think></think>and<an...",0.020660
2,Jamal saw a 380-foot whale with 8 57-inch remoras attached to it. What percentage of the whale's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 57 inches/remora * 8 remoras = 456 inches.\nThen divide that number by 12 to convert it to feet: 456 inches / 12 inches/foot = 38 foot.\nThen divide the combined remora length in feet by the whale's length and multiply by 100% to express ...","<|im_start|>systemAconversationbetweenUserandAssistant.Theuserasksaquestion,andtheAssistantsolvesit.Theassistantfirstthinksaboutthereasoningprocessinthemindandthenprovidestheuserwiththeanswer,andputyourfinalanswerwithin\boxed{{}}.Thereasoningprocessandanswerareenclosedwithin<think></think>and<an...",0.020660



--- Qwen_Qwen3.5-0.8B__float16__bf16__lr2e-06__steps20__maxc1024 ---
MultiArith accuracy: 0.2667 (4/15)
GSM-Symbolic avg TF-IDF score: 0.5271
Example outputs:


,question,formatted_question,answer,prediction,correct
0,"Paige had 11 songs on her mp3 player. If she deleted 9 old songs from it and then added 8 new songs, how many songs does she have on her mp3 player?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",10,"system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",True
1,"There were 39 girls and 4 boys trying out for the schools basketball team. If only 26 of them got called back, how many students didn't make the cut?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",17,"system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",False
2,"Faye had 34 coloring books. If she gave away 3 of them, but then bought 48 more, how many would she have total?","<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...",79,"system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",True


,question,formatted_question,answer,prediction,tfidf_score
0,Rania saw a 210-foot whale with 7 72-inch remoras attached to it. What percentage of the whale's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 72 inches/remora * 7 remoras = 504 inches.\nThen divide that number by 12 to convert it to feet: 504 inches / 12 inches/foot = 42 foot.\nThen divide the combined remora length in feet by the whale's length and multiply by 100% to express ...","system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",0.529875
1,Olivia saw a 10-foot dolphin with 8 3-inch remoras attached to it. What percentage of the dolphin's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 3 inches/remora * 8 remoras = 24 inches.\nThen divide that number by 12 to convert it to feet: 24 inches / 12 inches/foot = 2 foot.\nThen divide the combined remora length in feet by the dolphin's length and multiply by 100% to express th...","system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",0.537106
2,Jamal saw a 380-foot whale with 8 57-inch remoras attached to it. What percentage of the whale's body length is the combined length of the remoras?,"<|im_start|>system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process...","First, find the combined length of the remoras in inches: 57 inches/remora * 8 remoras = 456 inches.\nThen divide that number by 12 to convert it to feet: 456 inches / 12 inches/foot = 38 foot.\nThen divide the combined remora length in feet by the whale's length and multiply by 100% to express ...","system\nA conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer, and put your final answer within \boxed{{}} . The reasoning process and answer ...",0.517750
